# Bước 1 — Khảo sát và xác minh nguồn dữ liệu

**Đồ án:** Xây dựng mô hình phân loại tin tức tiếng Việt bằng Machine Learning và Deep Learning.

**Thời điểm khảo sát:** 20/11/2026 (Asia/Ho_Chi_Minh).

**Phạm vi:** chỉ kiểm tra robots.txt, RSS, sitemap và trang danh mục. Notebook **không crawl nội dung bài viết**, không làm sạch dữ liệu và chưa tạo `news_dataset.csv`.

**Cách dùng**
- Mặc định `RUN_LIVE = False`: đọc bằng chứng đã lưu trong `outputs/step1_source_audit.csv`.
- Đặt `RUN_LIVE = True` nếu muốn kiểm tra lại toàn bộ 90 endpoint. Quá trình này mất khoảng 3–5 phút vì có nghỉ 1,5 giây giữa các request.

In [1]:
from pathlib import Path
import subprocess
import sys
import pandas as pd

RUN_LIVE = False

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
AUDIT_PATH = PROJECT_ROOT / "outputs" / "step1_source_audit.csv"
SCRIPT_PATH = PROJECT_ROOT / "step1_khao_sat.py"

if RUN_LIVE:
    subprocess.run([sys.executable, str(SCRIPT_PATH)], check=True)

audit = pd.read_csv(AUDIT_PATH, encoding="utf-8-sig")
print("Đã đọc:", AUDIT_PATH)
print("Số dòng:", len(audit))
audit.head()

Đã đọc: /Users/mainguyen/Documents/Deepseek Harness/vn_news_classification/outputs/step1_source_audit.csv
Số dòng: 40


,Label,Source,Original_Category,RSS_URL,RSS_HTTP,RSS_Items,RSS_Unique_Links,RSS_Date_Min,RSS_Date_Max,Category_URL,Category_HTTP,Category_Article_Links,Selected_Method,Candidate_Quota,Notes
0,Kinh tế,VnExpress,Kinh doanh,https://vnexpress.net/rss/kinh-doanh.rss,200,60,60,2026-11-17T08:14:20+07:00,2026-11-20T11:11:51+07:00,https://vnexpress.net/kinh-doanh,200,50,category_pages,150,Dùng trang danh mục p1–p5 (p1 50 link; p2–p5 k...
1,Kinh tế,TuoiTre,Kinh doanh,https://tuoitre.vn/kinh-doanh.rss,200,50,50,2026-11-19T15:05:00+07:00,2026-11-20T12:45:00+07:00,https://tuoitre.vn/kinh-doanh.htm,200,14,rss,50,Lấy 50 URL từ RSS; ngày RSS dùng định dạng M/D...
2,Kinh tế,ThanhNien,Kinh tế,https://thanhnien.vn/rss/kinh-te.rss,200,50,50,2026-11-19T06:03:00+07:00,2026-11-20T12:00:00+07:00,https://thanhnien.vn/kinh-te.htm,200,53,rss,50,Lấy 50 URL từ RSS. Một số bài du lịch cũ nhất ...
3,Kinh tế,DanTri,Kinh doanh,https://dantri.com.vn/rss/kinh-doanh.rss,200,100,100,2026-11-17T13:16:55+07:00,2026-11-20T12:13:52+07:00,https://dantri.com.vn/kinh-doanh.htm,200,50,rss,100,Lấy 100 URL từ RSS. KHÔNG dùng phân trang /tra...
4,Kinh tế,VietnamNet,Kinh doanh,https://vietnamnet.vn/rss/kinh-doanh.rss,200,1000,1000,2026-10-04T05:30:00+07:00,2026-11-20T11:00:16+07:00,https://vietnamnet.vn/kinh-doanh,200,31,rss,400,"RSS trả 1.000 mục, đủ dùng; lấy 400 URL mới nh..."


## 1. Số mục trong RSS tại thời điểm khảo sát

Đây là số `<item>` thô. Một số mục có thể là bài PR, video, ảnh, hoặc nội dung không phù hợp; chưa phải số bài hợp lệ cuối cùng.

In [2]:
rss_pivot = audit.pivot(index="Label", columns="Source", values="RSS_Items")
rss_pivot = rss_pivot.loc[["Kinh tế", "Thế giới", "Thể thao", "Pháp luật",
                           "Giáo dục", "Sức khỏe", "Công nghệ", "Du lịch"]]
rss_pivot

Source,DanTri,ThanhNien,TuoiTre,VietnamNet,VnExpress
Label,,,,,
Kinh tế,100,50,50,1000,60
Thế giới,100,50,50,1000,60
Thể thao,100,50,50,1000,60
Pháp luật,100,50,50,1000,60
Giáo dục,100,50,50,1000,60
Sức khỏe,100,50,50,1000,60
Công nghệ,100,50,50,36,60
Du lịch,100,50,50,1000,60


## 2. Trạng thái endpoint

Tất cả 40 feed và 40 trang danh mục phải trả HTTP 200. Trang danh mục chỉ được kiểm tra để xác minh đường dẫn, chưa dùng để crawl hàng loạt.

In [3]:
summary = pd.DataFrame({
    "RSS_200": audit.groupby("Source")["RSS_HTTP"].apply(lambda s: int((s == 200).sum())),
    "Category_200": audit.groupby("Source")["Category_HTTP"].apply(lambda s: int((s == 200).sum())),
    "RSS_rows": audit.groupby("Source").size(),
})
summary

,RSS_200,Category_200,RSS_rows
Source,,,
DanTri,8,8,8
ThanhNien,8,8,8
TuoiTre,8,8,8
VietnamNet,8,8,8
VnExpress,8,8,8


## 3. Điểm bất thường đã phát hiện

- VietnamNet Công nghệ: feed chỉ còn 36 mục và dừng cập nhật từ 06/05/2025.
- VnExpress Khoa học công nghệ: trang 1 bao gồm cả tin khoa học; từ trang 2 URL chuyển sang `/cong-nghe-pN`. Cần kiểm tra nhãn thủ công.
- Tuổi Trẻ dùng ngày RSS dạng `M/D/YYYY h:mm:ss AM/PM`, không phải RFC 822.
- Dân Trí chặn phân trang `/trang-1..9` trong robots.txt.
- VnExpress chặn user-agent `Scrapy`; hệ thống dùng `requests` với UA minh bạch.

In [4]:
tech = audit[audit["Label"] == "Công nghệ"][
    ["Source", "Original_Category", "RSS_Items", "RSS_Date_Min", "RSS_Date_Max",
     "Selected_Method", "Candidate_Quota"]
].reset_index(drop=True)
tech

,Source,Original_Category,RSS_Items,RSS_Date_Min,RSS_Date_Max,Selected_Method,Candidate_Quota
0,VnExpress,Khoa học công nghệ,60,2026-11-16T00:00:00+07:00,2026-11-20T11:04:51+07:00,category_pages,150
1,TuoiTre,Nhịp sống số,50,2026-09-30T07:55:00+07:00,2026-11-17T15:19:00+07:00,rss,50
2,ThanhNien,Công nghệ,50,2026-11-16T07:46:00+07:00,2026-11-20T11:23:00+07:00,rss,50
3,DanTri,Công nghệ,100,2026-11-03T17:48:01+07:00,2026-11-20T08:00:00+07:00,rss,100
4,VietnamNet,Công nghệ,36,2025-02-03T13:23:08+07:00,2025-05-06T17:11:33+07:00,category_pages,400


## 4. Kế hoạch lấy URL đề xuất (chưa thực thi)

Mục tiêu tạm tính: **750 URL ứng viên/lĩnh vực**, tổng **6.000 URL**. Đây là số ứng viên trước khi loại bài lỗi/trùng; số bài hợp lệ kỳ vọng khoảng 600–700/lĩnh vực.

In [5]:
plan = (audit.groupby("Label", sort=False)
        .agg(Candidate_Total=("Candidate_Quota", "sum"),
             Sources=("Source", "nunique"))
        .reset_index())
assert set(plan["Candidate_Total"]) == {750}
assert len(audit) == 40
assert (audit["RSS_HTTP"] == 200).all()
assert (audit["Category_HTTP"] == 200).all()
plan

,Label,Candidate_Total,Sources
0,Kinh tế,750,5
1,Thế giới,750,5
2,Thể thao,750,5
3,Pháp luật,750,5
4,Giáo dục,750,5
5,Sức khỏe,750,5
6,Công nghệ,750,5
7,Du lịch,750,5


## 5. Kết luận

1. Cả 5 báo đều truy cập được và cho phép các đường dẫn cần dùng theo robots.txt.
2. 40/40 feed và 40/40 trang danh mục trả HTTP 200.
3. Có thể lấy đủ 8 lĩnh vực bằng RSS + trang danh mục, không cần sitemap cho kế hoạch chính.
4. VietnamNet Công nghệ là ngoại lệ bắt buộc: dùng trang danh mục thay cho RSS đã cũ.
5. Bước tiếp theo chỉ được bắt đầu sau khi người dùng xác nhận.

**Chưa thực hiện:** crawl nội dung, làm sạch, gán nhãn cuối, tạo CSV dataset.